# Mini NURA — Healthcare RAG Assistant
Answers questions using **only** the WHO fact sheets in `docs/`. Refuses out-of-scope questions and shows the source document for every answer.

## Install dependencies
Run once, then comment out.

In [9]:
# %pip install anthropic chromadb sentence-transformers python-dotenv

## Imports and API key

In [10]:
import os
import re
import glob
import time
from pathlib import Path
from dotenv import load_dotenv
from google import genai

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Add it to your .env file.")

client = genai.Client(api_key=GEMINI_API_KEY)
print("Imports OK. Gemini API key loaded.")

Imports OK. Gemini API key loaded.


## Load and chunk documents

In [11]:
DOCS_DIR   = Path("docs")
CHUNK_SIZE = 400   # characters per chunk
CHUNK_OVERLAP = 50 # overlap between consecutive chunks


def load_documents(docs_dir: Path) -> list[dict]:
    """Read all .txt files from docs/. Returns list of {filename, source_url, text}."""
    docs = []
    for path in sorted(docs_dir.glob("*.txt")):
        raw = path.read_text(encoding="utf-8").strip()
        # Extract source URL from first line if present
        lines = raw.splitlines()
        source_url = ""
        if lines and lines[0].startswith("SOURCE:"):
            source_url = lines[0].replace("SOURCE:", "").strip()
            raw = "\n".join(lines[2:]).strip()  # skip SOURCE line + blank line
        docs.append({
            "filename": path.stem,           # e.g. 'diabetes'
            "source_url": source_url,
            "text": raw,
        })
    return docs


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks, breaking on sentence boundaries where possible."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Try to break at a sentence boundary (. or \n) within the last 80 chars
        if end < len(text):
            boundary = text.rfind('.', start + chunk_size - 80, end)
            if boundary == -1:
                boundary = text.rfind('\n', start + chunk_size - 80, end)
            if boundary != -1:
                end = boundary + 1
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap
    return [c for c in chunks if len(c) > 30]  # drop tiny trailing chunks


def build_all_chunks(docs: list[dict]) -> tuple[list[str], list[str], list[dict]]:
    """Returns parallel lists: ids, texts, metadatas — ready for ChromaDB."""
    ids, texts, metadatas = [], [], []
    for doc in docs:
        chunks = chunk_text(doc["text"])
        for i, chunk in enumerate(chunks):
            ids.append(f"{doc['filename']}_chunk_{i}")
            texts.append(chunk)
            metadatas.append({
                "filename": doc["filename"],
                "source_url": doc["source_url"],
                "chunk_index": i,
                "total_chunks": len(chunks),
            })
    return ids, texts, metadatas


docs = load_documents(DOCS_DIR)
ids, texts, metadatas = build_all_chunks(docs)

print(f"Loaded {len(docs)} document(s)")
for doc in docs:
    doc_chunks = [t for i, t in enumerate(texts) if metadatas[i]['filename'] == doc['filename']]
    placeholder = '[Paste' in doc['text']
    status = '⚠ PLACEHOLDER — paste real content' if placeholder else f'{len(doc_chunks)} chunks'
    print(f"  {doc['filename']:20s} → {status}")
print(f"\nTotal chunks: {len(texts)}")

Loaded 5 document(s)
  Asthma               → 23 chunks
  Botulism             → 38 chunks
  Hantavirus           → 26 chunks
  Tetanus              → 20 chunks
  YellowFever          → 17 chunks

Total chunks: 124


## Build ChromaDB index
Skips automatically if the index already exists. Delete `chroma_db/` to rebuild.

In [12]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_PATH  = "chroma_db"
COLLECTION   = "who_facts"
EMBED_MODEL  = "all-MiniLM-L6-v2"  # fast, local, no API cost

# Embedding function (runs locally via sentence-transformers)
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL
)

# Persistent client — survives kernel restarts
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Check if collection already populated
existing = chroma_client.list_collections()
existing_names = [c.name for c in existing]

if COLLECTION in existing_names:
    collection = chroma_client.get_collection(name=COLLECTION, embedding_function=embed_fn)
    count = collection.count()
    print(f"Index already exists with {count} chunks — skipping re-embed.")
    print("Delete chroma_db/ and re-run this cell to rebuild from scratch.")
else:
    print("Building index... (first run, may take ~30s)")
    collection = chroma_client.create_collection(
        name=COLLECTION,
        embedding_function=embed_fn,
        metadata={"hnsw:space": "cosine"},  # cosine similarity for semantic search
    )
    # Add in batches of 100 to avoid memory spikes
    BATCH = 100
    for i in range(0, len(ids), BATCH):
        collection.add(
            ids=ids[i:i+BATCH],
            documents=texts[i:i+BATCH],
            metadatas=metadatas[i:i+BATCH],
        )
        print(f"  Embedded {min(i+BATCH, len(ids))}/{len(ids)} chunks...", end="\r")
    print(f"\nIndex built. {collection.count()} chunks stored in {CHROMA_PATH}/")

Index already exists with 124 chunks — skipping re-embed.
Delete chroma_db/ and re-run this cell to rebuild from scratch.


## Retrieval function

In [13]:
TOP_K = 3                   # number of chunks to retrieve per query
SIMILARITY_THRESHOLD = 0.55 # cosine distance below this = out of scope
                            # (ChromaDB returns distance, lower = more similar)


def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """
    Retrieve the top_k most relevant chunks for a query.
    Returns list of {text, filename, source_url, chunk_index, distance}.
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text": doc,
            "filename": meta["filename"],
            "source_url": meta["source_url"],
            "chunk_index": meta["chunk_index"],
            "distance": round(dist, 4),
        })
    return chunks


def is_out_of_scope(chunks: list[dict]) -> bool:
    """Return True if the best match is too dissimilar to be reliable."""
    if not chunks:
        return True
    return chunks[0]["distance"] > SIMILARITY_THRESHOLD


print("Retrieval functions ready.")
print(f"Using top-{TOP_K} chunks, out-of-scope threshold: distance > {SIMILARITY_THRESHOLD}")

Retrieval functions ready.
Using top-3 chunks, out-of-scope threshold: distance > 0.55


## Cell 6 — RAG answer function (Claude + refusal logic)

In [14]:
SYSTEM_PROMPT = """You are NURA, a healthcare information assistant.
You answer questions ONLY using the document excerpts provided in the user message.

Rules you must follow:
1. Base your answer solely on the provided excerpts. Do not use outside knowledge.
2. If the excerpts do not contain enough information to answer the question, say so clearly and politely. Do not guess.
3. Keep answers concise and factual.
4. Do not provide personal medical advice or diagnoses."""

RATE_LIMIT_DELAY = 4  # seconds between API calls (~15 req/min, within free tier)


def build_context(chunks: list[dict]) -> str:
    """Format retrieved chunks into a readable context block for Claude."""
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[Excerpt {i} — Source: {chunk['filename']}.txt]")
        lines.append(chunk["text"])
        lines.append("")
    return "\n".join(lines).strip()


def ask_nura(question: str, verbose: bool = False) -> dict:
    """
    Full RAG pipeline: retrieve → check scope → generate answer.
    Returns {answer, sources, out_of_scope, chunks}.
    """
    chunks = retrieve(question)

    if verbose:
        print("Retrieved chunks:")
        for c in chunks:
            print(f"  [{c['filename']}] dist={c['distance']} — {c['text'][:80]}...")

    # Hard refusal for clearly out-of-scope questions
    if is_out_of_scope(chunks):
        return {
            "answer": (
                "I'm sorry, I can only answer questions based on the WHO fact sheets "
                "I have access to (diabetes, hypertension, tuberculosis, depression, "
                "and cancer). Your question doesn't appear to be covered by these documents."
            ),
            "sources": [],
            "out_of_scope": True,
            "chunks": chunks,
        }

    context = build_context(chunks)
    user_message = f"""Answer the following question using ONLY the excerpts below.
If the excerpts don't fully answer it, say so.

Question: {question}

Excerpts:
{context}"""

    time.sleep(RATE_LIMIT_DELAY)  # stay within free tier rate limit
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=user_message,
        config={"system_instruction": SYSTEM_PROMPT, "max_output_tokens": 512},
    )
    answer_text = response.text.strip()

    # Deduplicate sources
    seen = set()
    sources = []
    for c in chunks:
        if c["filename"] not in seen:
            seen.add(c["filename"])
            sources.append({"filename": c["filename"], "url": c["source_url"]})

    return {
        "answer": answer_text,
        "sources": sources,
        "out_of_scope": False,
        "chunks": chunks,
    }


print("NURA pipeline ready.")

NURA pipeline ready.


## Cell 7 — Display helper

In [15]:
def display_answer(question: str, result: dict):
    """Pretty-print a NURA result."""
    divider = "─" * 60
    print(divider)
    print(f"Q: {question}")
    print(divider)

    if result["out_of_scope"]:
        print(f"⚠  OUT OF SCOPE")
        print(result["answer"])
    else:
        print(result["answer"])
        print()
        print("Sources:")
        for s in result["sources"]:
            url = s["url"] or "(no URL)"
            print(f"  • {s['filename']}.txt — {url}")

    print(divider)


print("Display helper ready.")

Display helper ready.


## Cell 8 — Single question (good for demos)
Change the question below and re-run.

In [16]:
question = "What are the common symptoms of asthma and what triggers can make them worse?"

result = ask_nura(question)
display_answer(question, result)

────────────────────────────────────────────────────────────
Q: What are the common symptoms of asthma and what triggers can make them worse?
────────────────────────────────────────────────────────────
The common symptoms of asthma include:

*   A persistent cough (especially at night)
*   Wheezing (when exhaling, and sometimes when inhaling)
*   Shortness of breath or difficulty breathing (sometimes even when resting)
*   Chest tightness, which makes it difficult to breathe deeply

The provided excerpts do not contain information regarding what specific triggers can make these symptoms worse.

Sources:
  • Asthma.txt — https://www.who.int/news-room/fact-sheets/detail/asthma
────────────────────────────────────────────────────────────


## Cell 9 — Out-of-scope refusal test
Demonstrates the refusal behaviour clearly for demos.

In [17]:
out_of_scope_questions = [
    "What is the best diet for losing weight?",
    "How do vaccines work?",
    "What are the symptoms of Alzheimer's disease?",
]

for q in out_of_scope_questions:
    result = ask_nura(q)
    display_answer(q, result)
    print()

────────────────────────────────────────────────────────────
Q: What is the best diet for losing weight?
────────────────────────────────────────────────────────────
⚠  OUT OF SCOPE
I'm sorry, I can only answer questions based on the WHO fact sheets I have access to (diabetes, hypertension, tuberculosis, depression, and cancer). Your question doesn't appear to be covered by these documents.
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
Q: How do vaccines work?
────────────────────────────────────────────────────────────
The provided excerpts do not contain information explaining how vaccines work.

Sources:
  • Tetanus.txt — https://www.who.int/news-room/fact-sheets/detail/tetanus
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
Q: What are the symptoms of Alzheimer's disease?
────────────────────────────────────────────────────────────


## Cell 10 — Interactive Q&A loop
Type questions in the input box. Type `quit` or `exit` to stop.

In [18]:
# print("NURA Healthcare Assistant")
# print("Documents loaded:", ", ".join(doc["filename"] for doc in docs))
# print("Type 'quit' to exit.\n")

# while True:
#     try:
#         question = input("Your question: ").strip()
#     except (EOFError, KeyboardInterrupt):
#         print("\nSession ended.")
#         break

#     if not question:
#         continue
#     if question.lower() in ("quit", "exit", "q"):
#         print("Goodbye!")
#         break

#     result = ask_nura(question)
#     display_answer(question, result)
#     print()